In [ ]:
import pandas as pd
import numpy as np
from matminer.featurizers.conversions import StrToComposition
from matminer.featurizers.composition import ElementProperty
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, f1_score

# 1. Загрузка проверенного датасета и отбор нетоксичных оксидов
df = pd.read_csv("../data/oxides_dataset_v0.csv")
df_safe = df[df["is_nontoxic"] == True].copy().reset_index(drop=True)
print(f"Нетоксичных составов для обучения: {len(df_safe)}")

# 2. Генерация 132 дескрипторов Magpie по химической формуле
print("Расчет Magpie-признаков (займет 1–3 минуты)...")
df_safe = StrToComposition().featurize_dataframe(df_safe, "reduced_formula", ignore_errors=True)
ep_featurizer = ElementProperty.from_preset(preset_name="magpie")
df_safe = ep_featurizer.featurize_dataframe(df_safe, col_id="composition", ignore_errors=True)

feature_cols = ep_featurizer.feature_labels()
X = df_safe[feature_cols].fillna(0)
y = df_safe["target_ferro"]          # 1 = FM/FiM (есть намагниченность), 0 = AFM/NM
groups = df_safe["chemsys"]          # Разбиение строго по семействам составов

# 3. Базовые модели по плану: Логистическая регрессия и Случайный лес
models = {
    "Logistic Regression": make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100, class_weight="balanced", random_state=42, n_jobs=-1
    )
}

gkf = GroupKFold(n_splits=5)

for name, model in models.items():
    aucs, f1s = [], []
    for train_idx, test_idx in gkf.split(X, y, groups=groups):
        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        proba = model.predict_proba(X.iloc[test_idx])[:, 1]
        preds = model.predict(X.iloc[test_idx])
        aucs.append(roc_auc_score(y.iloc[test_idx], proba))
        f1s.append(f1_score(y.iloc[test_idx], preds))
    print(f"{name:20s} | ROC-AUC: {np.mean(aucs):.3f} ± {np.std(aucs):.3f} | F1: {np.mean(f1s):.3f}")

# Сохраняем признаки для следующего этапа (добавление структуры и проверка утечек)
df_safe.to_csv("../data/oxides_features_magpie.csv", index=False)
print("Файл oxides_features_magpie.csv сохранен.")